# Lab 09: Extracting Data from PDFs

*INFO 4617/5617 · Week 9 · Chapter 9, [Extracting Data from PDFs](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html)*

Today we code together. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over before the next one; click it to open it.

We'll take the City of Boulder's PDFs apart the way the chapter does: see what a page actually stores, sort pages into born-digital, scanned, and scanned with a text layer, pull text with pypdf and tables with pdfplumber, read a picture of a table with OCR, and check every extraction. We end with eight years of the city's sales and use tax, read from seven year-end revenue reports.

**The files:** the chapter's copies of the city's records, kept in the book's repository, so that a whole class doesn't hit the city's server at once. Only one step (Step 3) asks the city's portal itself, and in class only the instructor runs it.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Step 0 imports pypdf, pdfplumber, and pytesseract; if one stops with `ModuleNotFoundError`, run `%conda install -y -c conda-forge pypdf pdfplumber pytesseract tesseract` once in a new cell, as the chapter's "Installing pypdf, pdfplumber, and pytesseract" says, then restart the kernel.

On Friday, run every cell above where we stopped (**Run → Run All Above Selected Cell**).

## Step 0 — Set up, and get the first file (5 minutes)

Import what the lab uses, and define the chapter's `HEADERS` and `get_saved_copy(name)`: it downloads one of the book's saved files from `BOOK_DATA`, `https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-09/`, into a `pdfs` folder the first time you ask, and after that returns the copy on disk. A PDF is bytes, not text, so save [`response.content`](https://requests.readthedocs.io/en/latest/user/quickstart/#binary-response-content), not `response.text`. Then get the December 2024 revenue report and print the libraries' versions, and Tesseract's: `pytesseract` runs a separate program, Tesseract, and a missing program shows up here as `TesseractNotFoundError`. The chapter's "This Chapter's Files" has the code.

In [ ]:
# Import re, time, hashlib, Path (from pathlib), requests, pandas as pd, matplotlib.pyplot as plt, pypdf's PdfReader, pdfplumber, pytesseract
# Define BOOK_DATA, HEADERS (a User-Agent with your email, as in week 2), and PDF_DIR = Path("pdfs")
# Define get_saved_copy(name): if PDF_DIR / name doesn't exist, make the folder, download BOOK_DATA + name, and save the bytes
# Get "revenue-report-2024-12.pdf" as report_2024; print its size, the versions of pypdf, pdfplumber, pytesseract, and Tesseract
# Your code here

<details>
<summary>💬 Discuss</summary>

- In week 2 we put contact information in the User-Agent. Who reads this one, now that the request goes to GitHub rather than to the city?
- Week 6's exercises cached a page's raw HTML before parsing it. What does `get_saved_copy()` save us, and the server, when we rerun a cell twenty times?
- Why does the book serve its own copies instead of sending a class of forty to the city's records portal?

</details>

## Step 1 — What a PDF page draws (5 minutes)

A PDF page is a list of drawing instructions: characters at coordinates, lines and rectangles, and images. Nothing in it says "word", "line", or "table"; every tool infers those from position. [pdfplumber](https://github.com/jsvine/pdfplumber) lists every object on a page with its position in points (1/72 inch) from the left edge (`x0`) and the top (`top`). Look at page 2, where the report's tables start. Then read what the file says about itself with pypdf's [`metadata`](https://pypdf.readthedocs.io/en/stable/user/metadata.html), and whether it is *tagged*, with a structure tree for screen readers. The chapter's "What's Inside a PDF" has more, including the drawing instructions themselves.

In [ ]:
# Open report_2024 with pdfplumber; for page 2, print the page count and the page's width and height,
#   seven of page.chars (text, x0, top, fontname, size), and how many chars, lines, rects, and images it draws
# Open it with PdfReader as reader: print the metadata's title, subject, creator, and creation_date,
#   and whether reader.trailer["/Root"] holds "/StructTreeRoot" (a tagged PDF)
# Your code here

<details>
<summary>💬 Discuss</summary>

- The first characters spell "City of" one at a time, and the table's rules are hundreds of thin rectangles. What does a table finder have to rebuild before it can find a single cell?
- The subject of a report on December 2024 says March 2018. What else in a file's metadata would you hesitate to rely on, and what is the creator still good for?
- Week 4's XML and JSON store their structure in the file. What does a PDF store instead, and what does that cost the person who wants its data?

</details>

## Step 2 — What kind of PDF is it? (9 minutes)

Before extracting anything, sort the pages, because the right tool depends on how each was made. **Born-digital** pages have characters to extract; **scanned** pages are one image and no characters, so they need OCR; **scanned with a text layer** pages are one image plus text that someone's OCR already produced, of unknown quality. Write the chapter's `page_profile(path)`, which counts each page's characters of [`extract_text()`](https://pypdf.readthedocs.io/en/stable/user/extract-text.html) and its [`images`](https://pypdf.readthedocs.io/en/stable/user/extract-images.html), and run it on three files. Then ask the 2018 minutes who made them, and read their first lines. The chapter's "What Kind of PDF Is It?" has the routing table.

In [ ]:
# Define page_profile(path): for each page of PdfReader(path), count len(page.extract_text()) and len(page.images); return both lists
# Print the two lists for revenue-report-2024-12.pdf, revenue-report-2018.pdf, and council-minutes-2018-12-04.pdf
# For the 2018 minutes (minutes): print the metadata's creator and creation_date, and the first three lines of page 1's text
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

A short version of Additional Exercise 5's quality check.

**Do:** run `page_profile()` on one of two files the chapter mentions but doesn't profile, `council-minutes-2000-12-19.pdf` or `revenue-report-2021-12.pdf`, and give each page a route from the chapter's table: born-digital (no image), scanned (an image and next to no text), or an image with text (test the text layer). Print who made the file, too: its metadata's `creator` and `producer`.
**Share:** your file, how many pages take each route, and which pages you'd send to OCR.

In [ ]:
# Microsprint: profile one file and route each page: no image -> born-digital; an image and under 100 characters -> OCR;
#   an image and more text -> test the text layer. Count the pages on each route, list the OCR pages, and print creator and producer
# Your code here

<details>
<summary>💬 Discuss</summary>

- How many pages of your file go to OCR? Would one character count for the whole file have told you which ones?
- The scanner's own OCR turned "COUNCIL" into "COTJNCIL". Why is a text layer you didn't make something to test rather than something to trust?
- Week 8 began with three checks for whether a page needs a browser. Which of today's checks plays the part of View Source, and which the part of the Network tab?

</details>

## Step 3 — Where the files came from: the city's portal (6 minutes)

**In class, only the instructor runs this cell.** On October 6, 2026, the city's server stopped answering after 13 to 17 requests in a row; a room of us running it at once would look like one client sending forty. Write it now, and run it at home. Nothing later in the lab needs it, so before Friday's **Run All Above**, comment it out (select its lines and press Ctrl+/ or Cmd+/), or a room of us will send it at once after all.

The saved copies came from the city's Laserfiche WebLink portal. Its folder page is a shell that JavaScript fills in, and the browser's Network tab shows where the list comes from: a POST to `GetFolderListing2` that sends a folder's number as JSON and gets JSON back. The city's server leaves out an intermediate certificate, so Python's default check fails with `CERTIFICATE_VERIFY_FAILED`; pass the book's bundle with [`verify=bundle`](https://requests.readthedocs.io/en/latest/user/advanced/#ssl-cert-verification), never `verify=False`. Then download one report, after a pause, and check what came back before trusting it: the content type, the five bytes `%PDF-` that start every PDF, and the file's SHA-256 hash against the book's `manifest.csv`, its record of where each copy came from. The chapter's "Getting the Files" and "Appendix: Technical Notes" have the code.

In [ ]:
# In class, only the instructor runs this cell. Get boulder-ca-bundle.pem and manifest.csv with get_saved_copy(); read the manifest
# POST the folder page's JSON body (folder 194345) to PORTAL + "FolderListingService.aspx/GetFolderListing2" with verify=bundle;
#   print the folder's path, then each entry's entryId, name, and kind (type 0 is a folder; else data[1] is its page count)
# Sleep 30 s; GET ElectronicFile.aspx for docid 194397; print Content-Type, size, first 5 bytes; compare its SHA-256 with the manifest
# Your code here

<details>
<summary>💬 Discuss</summary>

- Week 8 found hidden APIs in the Network tab. Which parts of this request (the folder number, `LF8PROD2`, the body) would break first if the city upgraded its portal?
- Python's default check failed and the book's bundle passed. Why is `verify=False` the wrong fix, even for public records?
- The manifest records each file's address, retrieval time, and hash. What could a reader of your project check with it that they couldn't without it?

</details>

## Step 4 — Text two ways: pypdf and pdfplumber (9 minutes)

The minutes of December 5, 2024 were written in Word and saved as PDF, so every page has characters. [pypdf](https://pypdf.readthedocs.io/) writes them out in the order the page draws them; its `extraction_mode="layout"` places each piece by its position instead. [pdfplumber](https://github.com/jsvine/pdfplumber#extracting-text) starts from where each character ends up, grouping characters into words when the gap is under 3 points (`x_tolerance`) and words into lines when their tops line up. Print the top of page 1 all three ways and compare. The chapter's "Text with pypdf" shows, in the page's drawing instructions, why Word's first letters break off.

In [ ]:
# Open council-minutes-2024-12-05.pdf with PdfReader as minutes_2024; text = page 1's extract_text(); print its first 13 lines
# Print the first lines of extract_text(extraction_mode="layout"), skipping blank lines and shrinking runs of spaces to one
# Open the same file with pdfplumber and print the first 11 lines of page 1's extract_text()
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

A short version of Additional Exercise 4, on a table instead of prose.

**Do:** extract page 2 of `report_2024`, the page with the sales and use tax summary, all three ways, and print, from each, only the lines that hold `"Total Sales"` or `"current economic activity"`.
**Share:** whether the three extractions agree on these rows, and what a pattern that reads one row per line would take as the subtotal's label, "Total Sales & Use Tax based on current economic activity", which wraps onto two lines on the page.

In [ ]:
# Microsprint: extract page 2 of report_2024 with pypdf (plain and layout) and pdfplumber;
#   from each, print only the lines that contain "Total Sales" or "current economic activity"
# Your code here

<details>
<summary>💬 Discuss</summary>

- pypdf's plain mode broke lines. The chapter says layout mode breaks words instead ("C ham bers"), but ours didn't. What would you check before relying on either mode's output for a new file?
- In the microsprint all three tools agreed, and all three put the subtotal's figures on its label's second line. What would a one-row-per-line pattern call that row, and which check would notice?
- pypdf printed "P age 1" first, though the page number sits at the bottom. What does that tell you about the order a PDF is drawn in?

</details>

## Step 5 — Repair the text, then find things in it (9 minutes)

Repair what pypdf broke with [regular expressions](https://docs.python.org/3/library/re.html), matching what the extraction produced rather than what the page shows: a pattern for "Page 1" would miss "P age 1". Write the chapter's `clean_page_text(raw_text)`. It drops the page number, rejoins a first letter that Word drew apart ("M" and "ayor"), and collapses runs of spaces and tabs but **not** newlines, because a heading is recognizable by sitting on its own line. Then join and clean all eight pages, and run the chapter's five patterns with [`re.findall()`](https://docs.python.org/3/library/re.html#re.findall). The chapter's "Cleanup Strategies" and "Regular Expressions for PDF Text" explain each one.

In [ ]:
# Define clean_page_text(raw_text): re.sub away ^P ?age \d+ lines (MULTILINE); join \b([A-Z]{1,2})\n before a letter;
#   collapse [ \t]+ to one space; strip each line and drop empty ones. Print the first 8 cleaned lines of page 1 (text)
# Clean all eight pages joined by "\n" into full_text; with re.findall print: dates in words, dates with slashes,
#   the first and last clock times and how many, (Carried|Failed) n:n tallies, and dollar amounts
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

The chapter's advice, "check a repair written for one file on the next, and print what it changed", cut to five minutes.

**Do:** on every page of `council-minutes-2023-12-07.pdf`, find each place where `clean_page_text()`'s second rule (`\b([A-Z]{1,2})\n` before a letter) would join two lines. Count the joins. A join is suspicious when its capital letters don't start a word, that is, when the character before them isn't a space or a line break; print only those, as the word the join would make. (The other joins include the first names of members of the public who spoke, so print counts, not words.)
**Share:** how many joins, how many look suspicious, and which of those is really wrong.

In [ ]:
# Microsprint: for each page of council-minutes-2023-12-07.pdf, find every match of the join rule with re.finditer; count them
#   For a match whose capitals follow a character other than whitespace, print the page and the word the join makes
# Your code here

<details>
<summary>💬 Discuss</summary>

- The join rule was right every time on the December 5 minutes and wrong at least once here. When has a repair been tested enough to run on a hundred files?
- The tally pattern found two votes, both in the motion tables. How would you find out how many votes it missed, such as "by a vote of 9:0"?
- Week 7 dropped stopwords before counting words. What else would a count of a topic word like "housing" need before you compared one meeting with another?

</details>

## Step 6 — Tables with pdfplumber (11 minutes)

pdfplumber's [table finder](https://github.com/jsvine/pdfplumber#extracting-tables) rebuilds a grid from the page's rules and the sides of its rectangles (the *lines* strategy), and makes a cell wherever rules close a box. Run [`find_tables()`](https://github.com/jsvine/pdfplumber#extracting-tables) on the summary page, and check its first table against the page with `search()`: the totals row has no vertical rules beside it, so it falls outside the table, and nothing raises an error. Then fix it as the chapter does: give the columns yourself as `explicit_vertical_lines` (the left and right edges of the cells it did find), keep the page's horizontal rules, and crop just below the totals row. Last, turn the rows into a DataFrame with the chapter's `summary_frame()`, naming the columns by position and turning the figures into numbers with `pd.to_numeric()` once the dollar signs and commas are gone, as week 6 did with a scraped table. The chapter's "Finding Tables" lists every table setting.

In [ ]:
# Open report_2024 with pdfplumber as report; summary_page = page 2; print each found table's rounded bbox and row count
# default_rows = the first table's extract(); print its last row, and compare the table's bottom with the "Total Sales and Use Tax" row's
# Fix: explicit vertical lines at every cell's left and right edge, horizontal_strategy "lines", the page cropped to 395 points
# Define summary_frame(rows, columns, label="source") as the chapter does; build summary_df and print source and ytd_2024
# Your code here

### ⏱ Microsprint (6 minutes, alone or in pairs)

A short version of Additional Exercise 2, on another year. The chapter says the other reports' tables have different rules; does its fix carry over?

**Do:** open one other year-end report (`revenue-report-2018.pdf`, `-2019.pdf`, `-2020.pdf`, `-2021-12.pdf`, `-2022-12.pdf`, or `-2023-12.pdf`); its summary is on page 2, except 2020's, which is on page 3. Apply the fix, changing one thing: instead of cropping at 395 points, crop 3 points below the bottom of the `"Total Sales and Use Tax"` row that `search()` finds. Print the number of rows before and after, and the totals row.
**Share:** your year, whether the totals row came back, and its two figures (the year before, and this year).

In [ ]:
# Microsprint: for one other report's summary page, find_tables(); take the first table's cell edges as explicit vertical lines
#   crop from the table's top to 3 points below the "Total Sales and Use Tax" row from search(), and extract_table(settings)
#   print the rows before and after, and the non-empty cells of the last row
# Your code here

<details>
<summary>💬 Discuss</summary>

- The default finder dropped the totals row and raised nothing. What told us, and what would have told us if we hadn't looked at the page?
- Did the fix carry over to your year? Line up the class's totals: what do you notice about a year that two reports both print?
- Week 6's `pd.read_html()` turned an HTML table into a DataFrame in one call. What does HTML's `<td>` give you for free that a PDF table doesn't?

</details>

## Step 7 — No text layer: OCR (11 minutes)

*Optical character recognition* turns a picture of text into characters. [pytesseract](https://github.com/madmaze/pytesseract) runs Tesseract on a Pillow image, which pypdf hands you as `page.images[0].image`. First, page 7 of the December 2018 minutes, where the scanner's own text layer turned the motion into "CouNcrr, Mnunrn WnlvnR MovED To ADopr": read the image again and print the lines that hold "MOVED" or "PASSED". Then Exhibit 3 of the December 2024 report, a picture of a table under a note offering "a screen readable version" on request. Convert it to grayscale and read it with `config="--psm 6"`, the [page segmentation mode](https://tesseract-ocr.github.io/tessdoc/ImproveQuality.html#page-segmentation-method) for one uniform block of text. Parse each row with the chapter's `ROW` pattern, setting aside lines with a year that don't parse, and check each row: its twelve months should add up to its total. Here the chapter's loop becomes a function, `parse_ocr_rows(lines)`, so the microsprint can rerun it. The chapter's "When There Is No Text Layer: OCR" has the confidence scores too, and why they mislead.

In [ ]:
# OCR page 7's one image of the 2018 minutes (minutes, from Step 2) with image_to_string; print its size and the MOVED/PASSED lines
# exhibit_3 = page 15's image of report_2024, .convert("L"); ocr_lines = image_to_string(exhibit_3, config="--psm 6").splitlines()
# Define NUMBER, ROW (from the chapter), ocr_dollars(figure), and parse_ocr_rows(lines) -> (rows parsed, lines set aside)
# Print how many rows parsed and were set aside, the start of each set-aside line, and the largest gap between months and total
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

Change one parameter: the page segmentation mode.

**Do:** read `exhibit_3` again with `--psm 4` (one column of text of varying sizes) or `--psm 3` (Tesseract's default, which finds the layout itself), run `parse_ocr_rows()` on the lines, and print the number of lines, rows parsed, rows set aside, and the largest gap. Time it with `time.time()`.
**Share:** your mode, its three counts, its largest gap, and how many seconds it took.

In [ ]:
# Microsprint: for --psm 4 (or 3), OCR exhibit_3 again, parse the lines with parse_ocr_rows(), and print
#   the number of lines, rows parsed, rows set aside, the largest gap between months and total, and the seconds it took
# Your code here

<details>
<summary>💬 Discuss</summary>

- Which mode parsed the most rows? Would you choose a mode by that count alone, or by what the checks say about the rows it parsed?
- Every parsed row adds up within $2. Name an OCR error that a row's sum can't catch.
- Chapter 9 calls a picture of a table the extreme case of soft enclosure. In week 3's terms, who is fenced out by it, and what does it cost them to get back in?

</details>

## Step 8 — Check the extractions (7 minutes)

Extraction fails without errors, so write the checks as a function that returns a list of problems, the chapter's `check_summary(table)`: **count** the rows the page shows (nine), and check the **sums** (the five sources add up to the subtotal, and the subtotal and the two audit rows to the total, within $5 of rounding). Run it on the fixed table and on the default one. Then two more checks: **two documents agree** (OCR's 2024 total, from the picture of Exhibit 3, against the summary page's text), and **a sample coded by hand**: who was present at the December 5, 2024 roll call, measured as [precision and recall](https://en.wikipedia.org/wiki/Precision_and_recall) against the nine members the page names. The chapter's "Checking an Extraction" lists eight checks, cheapest first.

In [ ]:
# Define check_summary(table, tolerance=5): problems if not 9 rows or a total's label is missing; then, for each year,
#   if the five sources don't add up to the subtotal, or subtotal + the two audit rows don't add up to the total. Return the list
# Run it on summary_df and on default_df = summary_frame(default_rows, {"source": 0, "ytd_2023": 2, "ytd_2024": 4, "change": 5, "pct_change": 6})
# Compare OCR's TOTAL row for 2024 with summary_df's total; then precision and recall of the names after "present:" on page 1 (text)
# Your code here

<details>
<summary>💬 Discuss</summary>

- Precision was perfect and recall was 0.56. Which of the two would a quick look at the output have shown you, and why?
- The sums say we copied the city's figures faithfully. Why don't they say the figures are right?
- The chapter's AI parser, Docling, wrote a complete-looking Exhibit 3 with two rows under the wrong label. Which of today's checks would you run on a parser's table first, and which would miss a wrong label?

</details>

## Step 9 — Seven reports, one time series (10 minutes)

The chapter's case study: total sales and use tax from all seven year-end reports. The table's rules differ from year to year, but one row is all we need, so read it by position. `find_summary_page(pdf)` finds the page by the sentence it begins with ("This report provides"), not by its number, since 2020's is on page 3. `row_words(page, label)` uses [`search()`](https://github.com/jsvine/pdfplumber#the-pdfplumberpage-class) to find the label and crops a strip as tall as its line; `to_number()` reads "$134,538,892", "(1,145,528)", and "-$1,608,698". Each row gives two years, the year before and this one, and its own change column is a free check. Keep both years from every report, compare the years printed twice, and plot each year's latest printing. The chapter's "Case Study: Boulder Revenue Reports" has the code and the figure.

In [ ]:
# Define find_summary_page(pdf), row_words(page, label), and to_number(text) as the chapter does
# For each of the seven REPORTS: the two years from search(r"(20\d\d) +(20\d\d)"), and the totals row's first three figures;
#   print any report whose change column doesn't check; keep both years' totals in a DataFrame, totals
# Show the years printed twice and how far apart they are; keep each year's latest printing; plot it as bars, labeled with the change
# Your code here

<details>
<summary>💬 Discuss</summary>

- Sales and use tax fell 4.76% in 2020 and rose 12.65% in 2021. What would you need before calling that a pandemic effect?
- The 2018 total is printed two ways, $38,959 apart. Which would you use, and what would you write in the data's notes?
- Week 3 named oversight as a value for public-interest data work. What can someone outside City Hall check with this series that they couldn't with the PDFs alone?

</details>

## Step 10 — What did we learn? (4 minutes)

In a few sentences, as comments or in a Markdown cell:

- Which pages we opened today needed OCR, and how could you tell without looking at them?
- Name one extraction that failed without an error today, and the check that caught it.
- How did Boulder's sales and use tax change from 2019 to 2024, and what lets you trust the figures enough to put eight years side by side?
- For a hundred council minutes, would you start from pypdf with repairs or from pdfplumber? What would you check first?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, collect three PDFs from one government source, politely, into `pdfs/`: Step 0's `get_saved_copy()` (User-Agent, save the bytes, download once) and Step 3's pause and `%PDF-` check.
- Step 2, open one and print its page count and page 1's raw text: Steps 1, 2, and 4.
- Step 3, write a cleanup function and show a passage before and after: Step 5's `clean_page_text()`, and its microsprint's check on a second file.
- Step 4, extract one element with `re.findall()`: Step 5's five patterns.
- Step 5, loop the pipeline over all three PDFs into a DataFrame: Step 9's loop over seven reports, and Step 2's `page_profile()` for page counts and characters.
- Step 6, plot your element's count per document and look at characters per page: Step 9's plot, and Step 2's routing.
- Step 7, interpret, and judge extraction quality: Step 8's checks.

In [ ]:
# Your answers here, as comments


## Friday: from the lab to the textbook (15 minutes)

What did the lab turn up that the chapter should know? Some places to look:

- **Outputs that changed.** In Step 4, pypdf's layout mode kept "Council Chambers" and "Carried" whole, where the chapter prints "C ham bers" and "Car r ied" and builds a paragraph on it ("Plain mode breaks lines, and layout mode breaks words"). What did yours print? With Tesseract 5.3.4, Step 7 parsed 29 of Exhibit 3's rows and set 1 aside; the chapter, with 5.5.3, reports 28 and 2. Did your copy match either?
- **A microsprint that would make a good exercise.** Step 6's microsprint cropped at the row `search()` found instead of at 395 points, and the fix worked on another year. Does it work on all seven? If so, it could become an Additional Exercise, or a sentence in "Finding Tables".
- **A sentence that confused you.** Did any of the chapter's explanations of pypdf's two modes, the join rule, or the table settings leave you guessing during the lab?
- **A check the chapter could add.** Step 5 showed the tally pattern finds votes only in the motion tables. Is there a short check that would count what it misses?

[`handouts/common/revision-framework.md`](../../handouts/common/revision-framework.md) says how to propose a revision.